(mmm_experiment_designer)=
# Experiment Designer: Posterior-Aware Lift Test Design

This notebook demonstrates the `ExperimentDesigner`, which recommends which marketing experiment to run — which channel, at what spend level, for how long — based on a fitted MMM's posterior uncertainty.

## Motivation

After fitting a Marketing Mix Model, practitioners often want to run a lift test to validate their model's predictions. But they face key design questions:

- **Which channel** should they test?
- **At what spend level** — where on the saturation curve should the test probe?
- **For how long** — what duration balances power against cost?
- **What is the expected lift**, given the model's uncertainty?

The `ExperimentDesigner` answers these questions using the MMM's posterior distribution. Instead of guessing the expected effect size (as tools like GeoLift require), it computes the **posterior-predicted lift** and **Bayesian assurance** (posterior-predictive power) for every candidate experiment.

In [ ]:
import matplotlib.pyplot as plt

from pymc_marketing.mmm.experiment_design import (
    ExperimentDesigner,
    generate_experiment_fixture,
)

%config InlineBackend.figure_format = "retina"

## Step 1: Load a Fixture (or Use a Fitted MMM)

The `ExperimentDesigner` can be created from:
- A **fitted MMM** via `ExperimentDesigner(mmm)` — the primary workflow
- A **saved InferenceData** via `ExperimentDesigner.from_idata(idata)` — for demos and testing

Here we generate a synthetic fixture with known ground-truth parameters for three channels:

| Channel | λ (saturation efficiency) | β (scale) | α (adstock decay) |
|---------|--------------------------|-----------|--------------------|
| TV      | 0.5 (slow saturation)     | 3.0       | 0.7 (slow decay)   |
| Search  | 2.0 (fast saturation)     | 1.5       | 0.3 (fast decay)   |
| Social  | 1.0 (moderate)            | 0.8       | 0.5 (moderate)     |

In [ ]:
idata = generate_experiment_fixture(
    channels=["tv", "search", "social"],
    true_params={
        "tv": {"lam": 0.5, "beta": 3.0, "alpha": 0.7},
        "search": {"lam": 2.0, "beta": 1.5, "alpha": 0.3},
        "social": {"lam": 1.0, "beta": 0.8, "alpha": 0.5},
    },
    fit_model=False,
    seed=42,
)

designer = ExperimentDesigner.from_idata(idata)
print(f"Channels: {designer.channel_columns}")
print(f"Posterior draws: {designer.n_draws}")
print(f"Adstock l_max: {designer.l_max}, normalize: {designer.normalize}")

## Step 2: Inspect Posterior Uncertainty

Before recommending experiments, let's visualise what the model believes about each channel. The `plot_channel_diagnostics()` method shows four key dimensions per channel:

1. **Posterior HDI width** — how uncertain are the saturation parameters?
2. **Mean spend correlation** — how correlated is this channel with others?
3. **Saturation gradient** — how responsive is the channel at the current operating point?
4. **Adstock α** — how quickly does the effect decay?

In [ ]:
fig, axes = designer.plot_channel_diagnostics()
fig.suptitle("Channel Diagnostics", fontsize=14, y=1.04)
plt.show()

## Step 3: Visualise Saturation Curves

The saturation curve shows how the channel's response changes with spend. The uncertainty band reflects posterior uncertainty in the saturation parameters. The vertical line marks the current operating point.

In [ ]:
fig, ax = designer.plot_saturation_curve("tv", n_samples=30, spend_levels=[0.2, 0.5])
plt.show()

## Step 4: Recommend Experiments

The `recommend()` method evaluates a grid of candidate experiments across all channels and returns a ranked list. Each candidate is defined by:
- **Channel** — which channel to test
- **Spend change** — fractional change to weekly spend (e.g., +20%, -50%, go-dark)
- **Duration** — experiment length in weeks

For each candidate, the designer computes:
- **Expected lift** — posterior mean of the cumulative lift over the experiment
- **Assurance** — Bayesian posterior-predictive power (probability of detecting the effect)
- **Adstock ramp fraction** — how much of the steady-state effect is captured
- **Net cost** — direct additional spend over the experiment duration
- **Composite score** — weighted combination of scoring dimensions

In [ ]:
recommendations = designer.recommend(
    spend_changes=[0.1, 0.2, 0.3, 0.5, -0.2, -0.5, -1.0],
    durations=[4, 6, 8, 12],
    min_snr=2.0,
    significance_level=0.05,
)

print(f"Total recommendations (passing min_snr filter): {len(recommendations)}")

### Recommendation Table

Let's display the top recommendations in a readable format:

In [ ]:
import pandas as pd

rows = []
for i, rec in enumerate(recommendations[:10]):
    rows.append(
        {
            "Rank": i + 1,
            "Channel": rec.channel,
            "Δ Spend": f"{rec.spend_change_frac:+.0%}/wk",
            "Duration": f"{rec.duration_weeks}w",
            "E[Lift]": f"{rec.expected_lift:.1f}",
            "Lift HDI": f"[{rec.expected_lift_hdi[0]:.0f}, {rec.expected_lift_hdi[1]:.0f}]",
            "SNR": f"{rec.snr:.1f}",
            "Assurance": f"{rec.assurance:.2f}",
            "Ramp": f"{rec.adstock_ramp_fraction:.2f}",
            "Score": f"{rec.score:.3f}",
        }
    )

df = pd.DataFrame(rows)
df

### Reading the Top Recommendation

Each recommendation includes an auto-generated rationale explaining why it was ranked where it is:

In [ ]:
top = recommendations[0]
print(top.rationale)

## Step 5: Diagnostic Plots

### Power vs. Cost

This scatter plot shows each candidate experiment's assurance (posterior-predictive power) against its absolute cost. Points are coloured by channel and shaped by spend direction (increase, decrease, go-dark).

In [ ]:
fig, ax = designer.plot_power_cost(recommendations)
plt.show()

### Lift Distributions

For a single channel, this grid shows the posterior distribution of total predicted lift across different spend changes and durations. The shaded region is the 94% HDI, and the dashed line is at zero.

In [ ]:
fig, axes = designer.plot_lift_distributions(
    "tv",
    spend_changes=[0.2, 0.5, -0.5, -1.0],
    durations=[4, 6, 8, 12],
)
plt.show()

### Adstock Ramp-up

The adstock ramp fraction shows how quickly each channel approaches its steady-state effect. Channels with high adstock α (slow decay) take longer to ramp up, meaning short experiments capture only a fraction of the true effect.

In [ ]:
fig, ax = designer.plot_adstock_ramp(max_weeks=16)
plt.show()

## Key Concepts

### Bayesian Assurance

Traditional power analysis picks a single effect size and computes $P(\text{detect} \mid \text{that effect})$. The `ExperimentDesigner` computes **posterior-predictive power** — the expected power averaged over the posterior distribution of the true effect:

$$\text{Assurance} = \mathbb{E}_{\theta \sim \text{posterior}}\left[\text{Power}(\theta)\right]$$

This produces intuitive behaviour:
- **Well-identified channels** with large effects → high assurance
- **Uncertain channels** → assurance reflects the risk that the true effect is too small to detect
- **Posterior includes zero** → pulls assurance down appropriately

### Adstock-Aware Lift

A spend change doesn't produce its full effect instantly. For slow-decaying channels (high α), the adstocked spend ramps toward the new steady state over many weeks. The designer accounts for this by computing the lift at each week of the experiment and summing. The **ramp fraction** quantifies how much of the steady-state effect is captured.

### Scoring

Experiments are ranked by a weighted composite score across five normalised dimensions:
1. **Posterior uncertainty** — prioritises channels where we'd learn the most
2. **Spend correlation** — prioritises channels contributing to the identification problem
3. **Saturation gradient** — prioritises channels where a spend change produces a larger response
4. **Assurance** — prioritises experiments likely to succeed
5. **Cost efficiency** — assurance per dollar of spend disruption

Weights are configurable via the `score_weights` parameter.

## Using with a Fitted MMM

In practice, you'd create the designer directly from a fitted `MMM`:

```python
from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.experiment_design import ExperimentDesigner

mmm = MMM(
    date_column="date",
    channel_columns=["tv", "search", "social"],
    target_column="revenue",
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
)
mmm.fit(X, y)

designer = ExperimentDesigner(mmm)
recommendations = designer.recommend()
```

The designer extracts everything it needs from the fitted model: posterior samples, current spend levels, residual noise, and spend correlations.